# 🌾 Crop AI - Siêu Huấn Luyện (Google Colab Edition)
Notebook này giúp huấn luyện mô hình nhận diện cây trồng với độ chính xác >90% sử dụng GPU.

In [ ]:
# 1. Kết nối Google Drive để lưu Model
from google.colab import drive
drive.mount('/content/drive')

# 2. Giải nén dữ liệu (Hãy upload file crop_data.zip lên thư mục gốc của Drive trước)
!cp /content/drive/MyDrive/crop_data.zip /content/
print("📦 Đang giải nén dữ liệu...")
!unzip -q -o /content/crop_data.zip -d /content/data_project

# 3. Quét và làm sạch ảnh lỗi bằng chính bộ giải mã của TensorFlow
import tensorflow as tf
import os
from pathlib import Path

print("🔍 Đang kiểm tra và loại bỏ các ảnh hỏng/không tương thích bằng bộ giải mã TF...")
data_dir = Path("/content/data_project/data")
deleted = 0
checked = 0

for root, _, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            continue
        checked += 1
        try:
            img_bytes = tf.io.read_file(file_path)
            # Đảm bảo ảnh giải mã ra 3 kênh RGB (3D Tensor) bình thường
            _ = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
        except Exception as e:
            print(f"❌ Xóa ảnh lỗi: {file_path} - {e}")
            try:
                os.remove(file_path)
                deleted += 1
            except Exception as del_err:
                print(f"   Khôg thể xóa: {del_err}")

print(f"✅ Hoàn tất dọn dẹp! Đã kiểm tra: {checked} ảnh, đã xóa: {deleted} ảnh lỗi.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Cấu hình
IMG_SIZE = (384, 384)
BATCH_SIZE = 32
DATA_DIR = "/content/data_project/data"
NUM_CLASSES = 38

# 1. Hàm tiền xử lý ảnh nâng cao (Đồng bộ với inference CLAHE + Denoise)
def apply_clahe_and_denoise_tf(image, label):
    def _preprocess_py(img_np):
        # Chuyển đổi sang dạng uint8 để xử lý OpenCV
        img_uint8 = np.clip(img_np, 0, 255).astype(np.uint8)
        
        # A. Median Blur khử nhiễu (Denoise)
        img_blur = cv2.medianBlur(img_uint8, 3)
        
        # B. CLAHE cân bằng tương phản sáng tối
        lab = cv2.cvtColor(img_blur, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        img_rgb = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        
        return img_rgb.astype(np.float32)

    # Sử dụng tf.py_function để bọc hàm Python/OpenCV chạy trên luồng dữ liệu của TF
    processed_img = tf.py_function(
        func=_preprocess_py,
        inp=[image],
        Tout=tf.float32
    )
    # Khôi phục shape cố định cho ảnh
    processed_img.set_shape((384, 384, 3))
    return processed_img, label

# Load Data bằng tf.keras.utils
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Áp dụng bộ lọc tiền xử lý nâng cao (CLAHE + Denoise)
train_ds = train_ds.unbatch().map(apply_clahe_and_denoise_tf, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
val_ds = val_ds.unbatch().map(apply_clahe_and_denoise_tf, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)

# Tối ưu hiệu năng tải dữ liệu
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# 2. Bộ tăng cường dữ liệu mạnh mẽ ngoài trời (Outdoor Data Augmentation)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.3),
    layers.RandomTranslation(0.15, 0.15),
    layers.RandomContrast(0.25),
    layers.RandomBrightness(0.2)
], name="data_augmentation")

# 3. Khởi dựng cấu trúc mô hình gốc bằng Keras 3 (tránh lỗi config)
print("🏗️ Đang khởi dựng mô hình EfficientNetV2S...")
base_model = tf.keras.applications.EfficientNetV2S(input_shape=(384, 384, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Đóng băng trong Phase 1

inputs = layers.Input(shape=(384, 384, 3))
x = data_augmentation(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dense(512, name="dense_512")(x)
x = layers.BatchNormalization(name="bn_1")(x)
x = layers.Activation("relu", name="relu_1")(x)
x = layers.Dropout(0.5, name="dropout_1")(x)
x = layers.Dense(256, name="dense_256")(x)
x = layers.BatchNormalization(name="bn_2")(x)
x = layers.Activation("relu", name="relu_2")(x)
x = layers.Dropout(0.25, name="dropout_2")(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name="predictions")(x)
model = models.Model(inputs, outputs, name="crop_super_v2s")

# Nạp trọng số từ file cũ bằng load_weights (bỏ qua lớp đầu ra mismatch và không lỗi định dạng lớp)
RESUME_PATH = "/content/drive/MyDrive/crop_super_v2s_best.h5"
if os.path.exists(RESUME_PATH):
    print(f"📦 Phát hiện mô hình cũ. Tiến hành nạp trọng số từ: {RESUME_PATH} ...")
    try:
        model.load_weights(RESUME_PATH, by_name=True, skip_mismatch=True)
        print("✅ Nạp trọng số thành công! Đã phục hồi kiến thức cũ.")
    except Exception as e:
        print(f"⚠️ Không thể nạp trọng số: {e}. Huấn luyện mới từ đầu (ImageNet).")
else:
    print("🆕 Không tìm thấy mô hình cũ trong Drive, huấn luyện từ đầu...")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Định nghĩa Callbacks thông minh (.keras format)
checkpoint_path = '/content/drive/MyDrive/crop_super_v2s_colab.keras'
my_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

# --- PHASE 1: HUẤN LUYỆN PHẦN HEAD (15 EPOCHS) ---
print("🚀 Bắt đầu Phase 1: Huấn luyện phần Head (Đóng băng backbone)...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=my_callbacks
)

# --- PHASE 2: FINE-TUNING TOÀN BỘ MÔ HÌNH (80 EPOCHS) ---
print("\n🚀 Bắt đầu Phase 2: Mở khóa các layer cuối và Fine-tuning...")
base_model.trainable = True
# Đóng băng các layer đầu, chỉ mở khóa 100 layer cuối
for layer in base_model.layers[:-100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=80,
    callbacks=my_callbacks
)

print(f"\n✅ ĐÃ HOÀN TẤT HUẤN LUYỆN! Model tốt nhất đã lưu tại Google Drive: {checkpoint_path}")

# 5. Vẽ đồ thị và lưu kết quả
def plot_and_save_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']
    
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    
    # Lưu vào Drive
    chart_path = '/content/drive/MyDrive/training_history_new.png'
    plt.savefig(chart_path)
    plt.show()
    print(f"📈 Biểu đồ lịch sử train đã được lưu vào Drive tại: {chart_path}")

plot_and_save_history(history_phase1, history_phase2)
